# Preparación del dataset para modelado binario FIRMS

Este notebook construye el dataset utilizado en `Modelos.ipynb`. El objetivo supervisado es estimar si una combinación departamento-semana presenta al menos una detección FIRMS.

Los tres perfiles nominales se forman únicamente con variables meteorológicas. `cantidad_detecciones` no participa en el escalado ni en K-Means; se utiliza después para construir y auditar el target binario. Nesterov tampoco entra al clustering ni se incorpora todavía al dataset definitivo.

## 1. Librerías y rutas

Se trabaja con la base semanal integrada existente. El resultado se guarda separado de la fuente original.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

RUTA_ENTRADA = Path("data/processed/firms_open_meteo_weekly/open_meteo_firms_weekly_target_available.parquet")
RUTA_SALIDA = Path("data/processed/modelado/dataset_modelado_presencia_firms.parquet")

CLAVES = ["departamento", "fecha_inicio_semana"]
VARIABLES_METEOROLOGICAS = [
    "temperature_2m_max_mean_weekly",
    "relative_humidity_2m_min_mean_weekly",
    "wind_speed_10m_max_mean_weekly",
    "precipitation_sum_weekly",
]
PERFILES = ["Cálido-seco", "Fresco-húmedo-lluvioso", "Fresco-seco"]

## 2. Carga y controles iniciales

Se comprueba que estén las columnas necesarias, que cada departamento-semana sea único y que FIRMS no tenga valores faltantes o negativos.

In [2]:
datos = pd.read_parquet(RUTA_ENTRADA)
datos["fecha_inicio_semana"] = pd.to_datetime(datos["fecha_inicio_semana"], errors="raise")

columnas_requeridas = set(CLAVES + VARIABLES_METEOROLOGICAS + ["cantidad_detecciones", "anio"])
columnas_faltantes = sorted(columnas_requeridas.difference(datos.columns))

assert not columnas_faltantes, f"Faltan columnas requeridas: {columnas_faltantes}"
assert not datos.duplicated(CLAVES).any(), "Hay claves departamento-semana duplicadas"
assert not datos["cantidad_detecciones"].isna().any(), "Hay detecciones faltantes"
assert not datos["cantidad_detecciones"].lt(0).any(), "Hay detecciones negativas"

print("Dimensiones de la fuente:", datos.shape)
print("Período:", datos["fecha_inicio_semana"].min().date(), "a", datos["fecha_inicio_semana"].max().date())
print("Claves duplicadas:", int(datos.duplicated(CLAVES).sum()))

Dimensiones de la fuente: (7923, 32)
Período: 2018-01-01 a 2025-12-22
Claves duplicadas: 0


## 3. Split temporal

Se conserva provisionalmente la partición existente: train 2018–2023, validation 2024 y test 2025. El test queda identificado, pero no se utiliza para evaluar modelos en este notebook. **PENDIENTE DE DEFINICIÓN METODOLÓGICA:** confirmar con la docente la estrategia final de evaluación supervisada. Mientras tanto, el ajuste del perfil usa solo train para no observar validation ni test.

In [3]:
datos["split"] = np.select(
    [
        datos["anio"].between(2018, 2023),
        datos["anio"].eq(2024),
        datos["anio"].eq(2025),
    ],
    ["train", "validation", "test"],
    default="fuera_alcance",
)

assert not datos["split"].eq("fuera_alcance").any()
datos["split"].value_counts().reindex(["train", "validation", "test"])

split
train         5947
validation    1007
test           969
Name: count, dtype: int64

## 4. Matriz meteorológica

K-Means utiliza solamente cuatro variables meteorológicas. Se aplica `log1p` a la precipitación para reducir su asimetría. El `StandardScaler` se ajusta exclusivamente con train para evitar fuga de información temporal.

In [4]:
def preparar_matriz_meteorologica(tabla):
    matriz = tabla[VARIABLES_METEOROLOGICAS].astype(float).copy()
    assert not matriz.isna().any().any(), "La matriz meteorológica contiene nulos"
    assert not matriz["precipitation_sum_weekly"].lt(0).any(), "La precipitación contiene valores negativos"
    matriz["precipitation_sum_weekly"] = np.log1p(matriz["precipitation_sum_weekly"])
    return matriz

es_train = datos["split"].eq("train")
matriz_train = preparar_matriz_meteorologica(datos.loc[es_train])
matriz_completa = preparar_matriz_meteorologica(datos)

scaler = StandardScaler()
matriz_train_escalada = scaler.fit_transform(matriz_train)
matriz_completa_escalada = scaler.transform(matriz_completa)

print("Variables utilizadas por K-Means:")
print(VARIABLES_METEOROLOGICAS)

Variables utilizadas por K-Means:
['temperature_2m_max_mean_weekly', 'relative_humidity_2m_min_mean_weekly', 'wind_speed_10m_max_mean_weekly', 'precipitation_sum_weekly']


## 5. Ajuste operativo de K-Means con tres perfiles

K-Means se ajusta únicamente con 2018–2023. Luego el modelo congelado asigna perfiles a todas las observaciones mediante `transform` y `predict`.

El análisis no supervisado con splits aleatorios seleccionó `k=3`. Para materializar la feature sin fuga respecto del split supervisado provisional, aquí se reajusta K-Means únicamente con train 2018–2023 y luego se aplica mediante `predict`. Los nombres son descripciones nominales, no niveles de riesgo.

In [5]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
kmeans.fit(matriz_train_escalada)

centroides = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=VARIABLES_METEOROLOGICAS,
)
centroides["precipitation_sum_weekly"] = np.expm1(centroides["precipitation_sum_weekly"])

calido_seco = centroides["temperature_2m_max_mean_weekly"].idxmax()
clusters_frescos = [cluster for cluster in centroides.index if cluster != calido_seco]
frio_humedo = centroides.loc[clusters_frescos, "precipitation_sum_weekly"].idxmax()
frio_seco = next(cluster for cluster in clusters_frescos if cluster != frio_humedo)

mapeo_perfiles = {
    int(calido_seco): "Cálido-seco",
    int(frio_humedo): "Fresco-húmedo-lluvioso",
    int(frio_seco): "Fresco-seco",
}

centroides["perfil_meteorologico"] = centroides.index.map(mapeo_perfiles)
centroides.sort_values("perfil_meteorologico")

,temperature_2m_max_mean_weekly,relative_humidity_2m_min_mean_weekly,wind_speed_10m_max_mean_weekly,precipitation_sum_weekly,perfil_meteorologico
1,28.030174,42.870919,5.514955,6.876984,Cálido-seco
2,20.200316,62.099735,5.978690,30.219065,Fresco-húmedo-lluvioso
0,18.544667,56.536042,4.873859,2.272997,Fresco-seco


In [6]:
clusters_asignados = kmeans.predict(matriz_completa_escalada)
datos["perfil_meteorologico"] = pd.Categorical(
    pd.Series(clusters_asignados, index=datos.index).map(mapeo_perfiles),
    categories=PERFILES,
)

pd.crosstab(datos["split"], datos["perfil_meteorologico"], dropna=False)

perfil_meteorologico,Cálido-seco,Fresco-húmedo-lluvioso,Fresco-seco
split,,,
test,296,379,294
train,1823,2296,1828
validation,246,523,238


## 6. Target binario y variables descriptivas

`presencia_firms` vale 1 si existe al menos una detección FIRMS y 0 en caso contrario. Se crean además mes y estación como variables de contexto. La cantidad original se conserva para trazabilidad, pero no se usa para formar los perfiles.

In [7]:
datos["presencia_firms"] = datos["cantidad_detecciones"].gt(0).astype("int8")
datos["mes"] = datos["fecha_inicio_semana"].dt.month.astype("int8")

estacion_por_mes = {
    12: "Verano", 1: "Verano", 2: "Verano",
    3: "Otoño", 4: "Otoño", 5: "Otoño",
    6: "Invierno", 7: "Invierno", 8: "Invierno",
    9: "Primavera", 10: "Primavera", 11: "Primavera",
}
datos["estacion"] = pd.Categorical(
    datos["mes"].map(estacion_por_mes),
    categories=["Verano", "Otoño", "Invierno", "Primavera"],
)

pd.crosstab(datos["split"], datos["presencia_firms"])

presencia_firms,0,1
split,,
test,613,356
train,3616,2331
validation,738,269


## 7. Dataset final

Se seleccionan explícitamente las columnas necesarias para el modelado y se ordenan las observaciones por departamento y semana. No se modifica el archivo integrado de origen.

In [8]:
columnas_salida = [
    "departamento", "fecha_inicio_semana", "anio", "mes", "estacion",
    *VARIABLES_METEOROLOGICAS,
    "perfil_meteorologico", "cantidad_detecciones", "presencia_firms", "split",
]

dataset_modelado = (
    datos[columnas_salida]
    .sort_values(CLAVES)
    .reset_index(drop=True)
)

dataset_modelado.head()

,departamento,fecha_inicio_semana,anio,mes,estacion,temperature_2m_max_mean_weekly,relative_humidity_2m_min_mean_weekly,wind_speed_10m_max_mean_weekly,precipitation_sum_weekly,perfil_meteorologico,cantidad_detecciones,presencia_firms,split
0,Artigas,2018-01-01,2018,1,Verano,30.764286,32.959184,6.566735,7.007143,Cálido-seco,0,0,train
1,Artigas,2018-01-08,2018,1,Verano,31.731633,44.836735,5.319592,32.842857,Cálido-seco,0,0,train
2,Artigas,2018-01-15,2018,1,Verano,29.744898,44.469388,5.492347,17.071429,Cálido-seco,0,0,train
3,Artigas,2018-01-22,2018,1,Verano,30.010204,52.051020,5.493673,25.335714,Cálido-seco,0,0,train
4,Artigas,2018-01-29,2018,1,Verano,31.524490,37.132653,5.431224,0.000000,Cálido-seco,0,0,train


## 8. Validaciones finales

Estos controles reemplazan al test separado: verifican dimensiones, claves, nulos, definición del target, perfiles y tamaños de los splits antes de guardar el archivo. Si algo cambia de forma inesperada, la ejecución se detiene con un mensaje claro.

In [9]:
assert dataset_modelado.shape == (7923, 13), f"Dimensión inesperada: {dataset_modelado.shape}"
assert not dataset_modelado.isna().any().any(), "El dataset final contiene valores faltantes"
assert not dataset_modelado.duplicated(CLAVES).any(), "El dataset final contiene claves duplicadas"
assert set(dataset_modelado["presencia_firms"].unique()) == {0, 1}
assert dataset_modelado["presencia_firms"].eq(
    dataset_modelado["cantidad_detecciones"].gt(0).astype("int8")
).all(), "El target binario no coincide con FIRMS"
assert set(dataset_modelado["perfil_meteorologico"].astype(str).unique()) == set(PERFILES)

splits_esperados = {"train": 5947, "validation": 1007, "test": 969}
assert dataset_modelado["split"].value_counts().to_dict() == splits_esperados

resumen_validacion = pd.DataFrame({
    "control": ["filas", "columnas", "duplicados", "nulos", "departamentos"],
    "resultado": [
        len(dataset_modelado),
        dataset_modelado.shape[1],
        int(dataset_modelado.duplicated(CLAVES).sum()),
        int(dataset_modelado.isna().sum().sum()),
        dataset_modelado["departamento"].nunique(),
    ],
})
resumen_validacion

,control,resultado
0,filas,7923
1,columnas,13
2,duplicados,0
3,nulos,0
4,departamentos,19


## 9. Exportación

La exportación se realiza únicamente después de superar todos los controles anteriores.

In [10]:
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
dataset_modelado.to_parquet(RUTA_SALIDA, index=False)

print("Dataset guardado en:", RUTA_SALIDA)
print("Dimensiones:", dataset_modelado.shape)
print("Mapeo cluster a perfil:", mapeo_perfiles)

Dataset guardado en: data/processed/modelado/dataset_modelado_presencia_firms.parquet
Dimensiones: (7923, 13)
Mapeo cluster a perfil: {1: 'Cálido-seco', 2: 'Fresco-húmedo-lluvioso', 0: 'Fresco-seco'}
